# TECLA · demostración

**Term Extraction and Corpus Linguistics Application**

Este cuaderno ejecuta la extracción de candidatos a término de TECLA sin instalar nada en tu ordenador. No hace falta saber programar: ejecuta las celdas en orden con el botón ▶ que aparece a la izquierda de cada una.

1. Instalar TECLA
2. Elegir el corpus
3. Extraer los candidatos a término
4. Ver el corpus
5. Descargar el resultado

> **Los textos que subas se copian a esta sesión de Google Colab.** No subas documentos confidenciales ni sujetos a derechos de autor. Para trabajar con tu propio corpus, [instala TECLA en tu ordenador](https://github.com/isabel-mm/TECLA): ahí los textos no salen de él. La evaluación de la representatividad del corpus y la revisión de los candidatos están en la aplicación de escritorio.

## 1. Instalar TECLA

Tarda unos dos minutos: descarga TECLA y los modelos lingüísticos de español e inglés.

In [ ]:
%%capture
!git clone -q https://github.com/isabel-mm/TECLA.git /content/TECLA
%cd /content/TECLA
!pip install -q -e .
!python -m spacy download es_core_news_md
!python -m spacy download en_core_web_lg

In [ ]:
import sys
REPO = "/content/TECLA"
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from pipeline.pipeline import run_tecla_pipeline
from tecla.extractor.tbx_export import export_tbx
print("TECLA está lista.")

## 2. Elegir el corpus

Puedes usar los **textos de ejemplo** (tres textos breves sobre lingüística de corpus, redactados para esta demostración) o subir **tus propios textos** en PDF o TXT.

In [ ]:
#@title Corpus { display-mode: "form" }
corpus = "Textos de ejemplo (inglés)"  #@param ["Textos de ejemplo (inglés)", "Textos de ejemplo (español)", "Mis textos"]

import shutil
from pathlib import Path

TRABAJO = Path("/content/mi_corpus")
CORPUS = TRABAJO / "corpus"
shutil.rmtree(TRABAJO, ignore_errors=True)
CORPUS.mkdir(parents=True)

if corpus == "Mis textos":
    from google.colab import files
    print("Selecciona tus archivos (PDF o TXT):")
    for nombre, contenido in files.upload().items():
        (CORPUS / nombre).write_bytes(contenido)
else:
    origen = Path(REPO) / "demo" / "ejemplos" / ("en" if "inglés" in corpus else "es")
    for f in sorted(origen.glob("*.txt")):
        shutil.copy(f, CORPUS / f.name)

print(f"{len(list(CORPUS.iterdir()))} archivo(s) en el corpus:")
for f in sorted(CORPUS.iterdir()):
    print("  ·", f.name)

## 3. Extraer los candidatos a término

TECLA limpia los textos, los analiza lingüísticamente, busca las secuencias que siguen los patrones gramaticales de los términos y calcula para cada una su *TF-IDF*, su *C-value* y su *NC-value*. Las dos medidas se combinan en un índice de terminologicidad, que ordena la lista.

El **perfil de puntuación** decide cuánto pesa el *TF-IDF* frente al *NC-value*: *Frecuencia* (0,6), *Equilibrado* (0,5) o *Complejidad* (0,2).

In [ ]:
#@title Parámetros { display-mode: "form" }
lengua = "Detección automática"  #@param ["Detección automática", "Español", "Inglés"]
incluir_monolexicos = True  #@param {type:"boolean"}
perfil = "Equilibrado"  #@param ["Frecuencia", "Equilibrado", "Complejidad"]

import pandas as pd

PERFILES = {"Frecuencia": {"tfidf": 0.6, "ncvalue": 0.4},
            "Equilibrado": {"tfidf": 0.5, "ncvalue": 0.5},
            "Complejidad": {"tfidf": 0.2, "ncvalue": 0.8}}
LENGUAS = {"Detección automática": None, "Español": "es", "Inglés": "en"}

documentos, detectada = [], []
resultados = run_tecla_pipeline(
    corpus_path=CORPUS,
    language=LENGUAS[lengua],
    cleaning_options={"structure": True, "remove_bibliography_flag": True,
                      "remove_captions_flag": True, "remove_section_headers_flag": True},
    extraction_options={"multiword_only": not incluir_monolexicos, "max_term_length": 5,
                        "weights": PERFILES[perfil]},
    on_corpus_ready=lambda docs, carpeta, lang: (documentos.extend(docs), detectada.append(lang)),
)
resultados.sort(key=lambda t: -t["terminologicity"])

candidatos = pd.DataFrame([{
    "Candidato": t["lemma"],
    "Variantes": " · ".join(t.get("variants", [])) or "—",
    "Terminologicidad": round(t["terminologicity"], 4),
    "TF-IDF": round(t.get("tf_idf", 0), 2),
    "C-value": round(t.get("c_value", 0), 2),
    "NC-value": round(t.get("nc_value", 0), 2),
    "Frecuencia": t["frequency"],
    "Documentos": t["doc_frequency"],
} for t in resultados])
candidatos.index += 1

print(f"\n{len(resultados)} candidatos a término. Estos son los 30 primeros:")
candidatos.head(30)

## 4. Ver el corpus

Datos descriptivos de cada documento: *tokens*, oraciones, tipos (lemas distintos) y relación entre tipos y *tokens* (TTR).

In [ ]:
filas, lemas, total = [], set(), 0
for d in documentos:
    tipos = {t["lemma"] for t in d["tokens"]}
    n = len(d["tokens"])
    lemas |= tipos
    total += n
    filas.append({"Documento": d["doc_id"], "Tokens": n, "Oraciones": len(d.get("sentences", [])),
                  "Tipos": len(tipos), "TTR": round(len(tipos) / n, 4) if n else 0})

print(f"{len(documentos)} documentos · {total} tokens · {len(lemas)} tipos")
pd.DataFrame(filas)

## 5. Descargar el resultado

La lista completa se guarda en dos formatos: una hoja de cálculo (CSV), con todas las cifras de cada candidato, y TBX-Basic (ISO 30042), el estándar de intercambio terminológico.

In [ ]:
import csv

ruta_csv = TRABAJO / "candidatos_tecla.csv"
ruta_tbx = TRABAJO / "candidatos_tecla.tbx"
campos = ["term", "lemma", "variants", "patterns", "terminologicity", "tf_idf",
          "c_value", "nc_value", "frequency", "freq_rel", "doc_frequency"]
with open(ruta_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=campos)
    w.writeheader()
    for t in resultados:
        fila = {k: t.get(k, "") for k in campos}
        fila["patterns"] = ";".join(t.get("patterns", []))
        fila["variants"] = ";".join(t.get("variants", []))
        w.writerow(fila)
export_tbx(resultados, ruta_tbx, language=(detectada[0] if detectada else "en"))

from google.colab import files
files.download(str(ruta_csv))
files.download(str(ruta_tbx))

---
TECLA se distribuye con licencia MIT. Código y documentación: [github.com/isabel-mm/TECLA](https://github.com/isabel-mm/TECLA)